In [1]:
!python -m pip install pandas numpy scipy matplotlib seaborn statsmodels jupyter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 23.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 36.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 950.3/950.3 kB 29.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [jupyter]


In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

In [3]:
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

pd.set_option('display.float_format',lambda x:f"{x:,.3f}")
sns.set_theme(style="whitegrid")
print("Libraries imported successfully")

Libraries imported successfully


We simulate 1,500 orders across three independently assigned marketing campaigns. The order_value variable is deliberately right-skewed (a lognormal distribution), which resembles spending data: many modest purchases and a small number of large purchases.

For the ANOVA exercise, we create campaign_spend using approximately normal distributions with different means and similar standard deviations. This gives us a clear example for learning one-way ANOVA assumptions and interpretation.

The two variables serve different teaching purposes:
- order_value: descriptive statistics, skewness, IQR, and boxplot analysis.
- campaign_spend: comparison of mean spending across three campaigns using ANOVA.
This separation is intentional: a strongly skewed variable may make classical ANOVA assumptions less convincing, so we should not blindly apply the same test to every variable.

In [4]:
n = 1500

# Right-skewed order values: lognormal data are positive and have a long right tail.
order_value = rng.lognormal(mean=4.45, sigma=0.55, size=n)

extreme_indices = rng.choice(n, size=8, replace=False)
order_value[extreme_indices] *= rng.uniform(3.0, 6.0, size=len(extreme_indices))

campaign = rng.choice(
    ['Campaign A', 'Campaign B', 'Campaign C'],
    size=n,
    replace=True
)

# Create a separate response variable with group means that differ.
campaign_means = {"Campaign A":82, "Campaign B":88, "Campaign C":95}
campaign_spend = np.array([
    rng.normal(campaign_means[c],12) for c in campaign
])

df = pd.DataFrame({
    "order_id":np.arange(10001,10001+n),
    "campaign":campaign,
    "order_value":order_value.round(2),
    "campaign_spend":campaign_spend.round(2),
    "deliverey_hours":rng.gamma(shape=5.0, scale=1.2,size=n).round(2),
    "items_per_order":rng.poisson(lam=3.2,size=n),
    "customer_rating":rng.integers(1, 6, size=n)
})

missing_idx = rng.choice(n, size=12, replace=False)
df.loc[missing_idx,"order_value"] = np.nan

df.head()

,order_id,campaign,order_value,campaign_spend,deliverey_hours,items_per_order,customer_rating
0,10001,Campaign C,101.250,98.980,5.100,8,2
1,10002,Campaign A,NaN,84.960,6.690,4,2
2,10003,Campaign C,129.380,108.020,4.530,5,2
3,10004,Campaign B,143.640,94.610,6.420,4,5
4,10005,Campaign B,29.280,59.470,5.430,7,5


In [5]:
print("Rows and Columns:",df.shape)
print("\\nData Types:")
print(df.dtypes)
print("\\nMissing Values:")
print(df.isna().sum())
print("\\nDuplicate Rows:",df.duplicated().sum())

Rows and Columns: (1500, 7)
\nData Types:
order_id             int64
campaign            object
order_value        float64
campaign_spend     float64
deliverey_hours    float64
items_per_order      int64
customer_rating      int64
dtype: object
\nMissing Values:
order_id            0
campaign            0
order_value        12
campaign_spend      0
deliverey_hours     0
items_per_order     0
customer_rating     0
dtype: int64
\nDuplicate Rows: 0


Interpretation checkpoint
- Each row represents one order (an observation).
- campaign is categorical.
- order_value, campaign_spend, and delivery_hours are numerical measurements.
- items_per_order is a discrete count.
- Missing values must be investigated before calculations. We will not silently replace them with zero: zero would mean a genuine zero-value order, which is not the same as an unknown value.

3. Data preparation for statistical analysis
For this project, we create an analysis copy and remove rows missing the variable being analysed. In a real project, decide how to handle missingness based on why values are missing and how much data is affected.

We also validate simple constraints. A negative order value or a rating outside 1–5 may indicate a data issue.

In [8]:
analysis_df = df.copy()

# Keep rows with observed order_value for order-value analysis.
order_analysis = analysis_df.dropna(subset=["order_value"]).copy()

# Basic validation checks.
assert (order_analysis["order_value"] > 0).all(), "Order values should be positive."
assert order_analysis["customer_rating"].between(1, 5).all(), "Rating outside 1–5."
assert order_analysis["items_per_order"].ge(0).all(), "Item count cannot be negative."

print(f"Original rows: {len(df)}")
print(f"Rows available for order-value analysis: {len(order_analysis)}")
print(f"Rows excluded because order_value was missing: {len(df) - len(order_analysis)}")

Original rows: 1500
Rows available for order-value analysis: 1488
Rows excluded because order_value was missing: 12


4. Descriptive statistics — definition, formulas, code, interpretation
Descriptive statistics are methods used to organize, summarize, and describe the main features of observed data. They do not, by themselves, prove causation or automatically generalize results to an entire population.
Measures of central tendency
- Mean: arithmetic average; sum of observations divided by the number of observations. It uses every value and is sensitive to extreme values.
- Median: middle value after sorting; for an even number of observations, the average of the two middle values. It is more resistant to extreme values.
- Mode: most frequently occurring value. A dataset may have one mode, multiple modes, or no useful unique mode.
Measures of spread
- Range: maximum minus minimum. It uses only two observations and is highly sensitive to extremes.
- Variance: average squared distance from the mean. Squaring prevents positive and negative deviations from cancelling. Sample variance uses n-1 in the denominator; population variance uses n.
- Standard deviation: square root of variance, expressed in the same units as the original variable.
- Quartiles: divide ordered data into four parts. Q1 is the 25th percentile, Q2 is the median (50th percentile), and Q3 is the 75th percentile.
- IQR: Q3 minus Q1; spread of the middle 50% of observations.
Sample versus population variance
For a population, divide the sum of squared deviations by N. For a sample, divide by n−1 (Bessel's correction), which corrects the bias of the sample variance as an estimator of population variance.

In [9]:
values = order_analysis["order_value"]

descriptive_summary = pd.Series({
    "count":values.count(),
    "mean":values.mean(),
    "median":values.median(),
    "mode":values.mode().iloc[0] if not values.mode().empty else np.nan,
    "std":values.std(),
    "minimum":values.min(),
    "Q1 (25th Percentile)":values.quantile(0.25),
    "Q2 (50th Percentile)":values.quantile(0.50),
    "Q3 (75th Percentile)":values.quantile(0.75),
    "IQR":values.quantile(0.75) - values.quantile(0.25),
    "maximum":values.max(),
    "range":values.max() - values.min(),
    "variance":values.var(),
    "skewness":values.skew(),
    "kurtosis":values.kurtosis(),
    "sample_variance":values.var(ddof=1),
    "sample_standard_deviation":values.std(ddof=1)
})

descriptive_summary.to_frame('values')


,values
count,"1,488.000"
mean,100.052
median,86.185
mode,86.170
std,61.606
minimum,11.510
Q1 (25th Percentile),59.193
Q2 (50th Percentile),86.185
Q3 (75th Percentile),121.558
IQR,62.365


How to interpret the output
Use the actual values printed by your notebook:
1. Compare mean and median. If mean is much larger than median, a high-value right tail may be pulling the mean upward.
2. Compare standard deviation with the mean to understand variability in context.
3. Range can be dominated by one extreme transaction; compare it with IQR.
4. Q1 and Q3 describe the central half of transactions. Half of observed values lie between them (subject to the quantile convention and ties).
5. Never say that the mean is always the “best” average. Choose a measure suited to the distribution and business question.


5. Calculate descriptive statistics manually
Cell 14 of 49

5. Calculate descriptive statistics manually
The purpose of this section is to understand what the library functions are doing. We will write small reusable functions and compare their results with Pandas.

For an observed sample x₁, …, xₙ:
- Mean = sum(xᵢ) / n
- Sample variance = sum((xᵢ − mean)²) / (n − 1)
- Sample standard deviation = square root of sample variance
The ddof=1 setting in NumPy requests sample variance/standard deviation. ddof=0 requests population variance/standard deviation.

In [10]:
def manual_mean(data):
    clean = [float(x) for x in data if pd.notna(x)]
    if len(clean) == 0:
        raise ValueError("Cannot calculate mean of empty data.")
    return sum(clean) / len(clean)


def manual_median(data):
    clean = sorted(float(x) for x in data if pd.notna(x))
    n = len(clean)
    if n == 0:
        raise ValueError("Cannot calculate median of empty data.")
    middle = n // 2
    if n % 2 == 1:
        return clean[middle]
    return (clean[middle - 1] + clean[middle]) / 2


def manual_sample_variance(data):
    clean = [float(x) for x in data if pd.notna(x)]
    n = len(clean)
    if n < 2:
        raise ValueError("At least two observations are required.")
    mean_value = sum(clean) / n
    return sum((x - mean_value) ** 2 for x in clean) / (n - 1)


def manual_sample_std(data):
    return manual_sample_variance(data) ** 0.5


demo = [10, 12, 14, 16, 18]
print("Manual mean:", manual_mean(demo))
print("Manual median:", manual_median(demo))
print("Manual sample variance:", manual_sample_variance(demo))
print("Manual sample standard deviation:", manual_sample_std(demo))

assert np.isclose(manual_mean(demo), np.mean(demo))
assert np.isclose(manual_median(demo), np.median(demo))
assert np.isclose(manual_sample_variance(demo), np.var(demo, ddof=1))
assert np.isclose(manual_sample_std(demo), np.std(demo, ddof=1))
print("Manual calculations match NumPy.")

Manual mean: 14.0
Manual median: 14.0
Manual sample variance: 10.0
Manual sample standard deviation: 3.1622776601683795
Manual calculations match NumPy.
